<a href="https://colab.research.google.com/github/younsai/efficientnet_training/blob/main/EfficientNet_CIFR10_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Loading Training and Test Datasets**

In [ ]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# **Importing the EfficientNet-B0 Model**

---

Loading model parameters

In [ ]:
import torchvision.models as models

weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

print("Modèle EfficientNet-B0 importé avec succès !")

Adapt the last layer to 10 classes

In [ ]:
import torch.nn as nn
import torch.optim as optim

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 10)

# **Training over 2 Epochs**
---

###**Training Configuration** :
####-Loss Function : CrossEntropy
####-Optimization Method : Adam

In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Modèle envoyé sur le périphérique : {device}")

###Starting Training

In [ ]:
print("4/4 - Démarrage de l'entraînement...")
for epoch in range(2):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Époque {epoch+1}/2 - Perte (Loss) : {running_loss/len(train_loader):.4f}")

print("Félicitations ! Entraînement terminé avec succès. 🎉")

# **Model Testing after Training**

---

In [ ]:
model.eval()
correct = 0
total = 0

# On désactive le calcul des gradients pour aller plus vite et économiser de la mémoire
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Prédiction du modèle
        outputs = model(images)

        # La classe prédite est celle qui a le score le plus élevé (le maximum)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Exactitude du modèle sur les images de test : {accuracy:.2f}%")

# **Practical Test : Image Classification**

---

In [ ]:
import os
import torch
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt

chemin_dossier = '/content/drive/MyDrive/efficientnetTest'

# List of CIFAR-10 classes
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

# Same transformations as for training
transformation_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Model configuration
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

extensions_valides = ('.jpg', '.jpeg', '.png', '.webp')

print("--- Starting image testing from Drive ---")

for nom_fichier in os.listdir(chemin_dossier):
    # Check if the file is an image
    if nom_fichier.lower().endswith(extensions_valides):
        chemin_complet = os.path.join(chemin_dossier, nom_fichier)

        try:
            # Load and transform the image
            image_brute = Image.open(chemin_complet).convert('RGB')
            image_transformee = transformation_test(image_brute).unsqueeze(0).to(device)

            # Prediction
            with torch.no_grad():
                outputs = model(image_transformee)
                probabilites = torch.nn.functional.softmax(outputs[0], dim=0)
                valeur_max, indice_predit = torch.max(probabilites, 0)

            # Graphic display
            plt.figure(figsize=(5, 5))
            plt.imshow(image_brute)
            plt.axis('off')

            classe_predite = classes[indice_predit.item()]
            confiance = valeur_max.item() * 100

            plt.title(f"File: {nom_fichier}\nPrediction: {classe_predite} ({confiance:.2f}%)",
                      fontsize=12, color="green" if confiance > 50 else "orange")
            plt.show()

        except Exception as e:
            print(f"Error reading {nom_fichier}: {e}")